In [2]:
# 1. Clone repository từ GitHub
!git clone https://github.com/tphan1420/PRDetect-Research.git

# 2. Chuyển vào thư mục làm việc của dự án
%cd PRDetect-Research

# 3. Chuyển sang nhánh chứa mã nguồn RGCN vừa nâng cấp
!git checkout feat/rgcn-multi-relational

# 4. Kiểm tra trạng thái Git
!git status


Cloning into 'PRDetect-Research'...
remote: Enumerating objects: 466, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 466 (delta 49), reused 69 (delta 24), pack-reused 371 (from 1)
Receiving objects: 100% (466/466), 162.61 MiB | 23.07 MiB/s, done.
Resolving deltas: 100% (164/164), done.
Updating files: 100% (248/248), done.
/kaggle/working/PRDetect-Research
Branch 'feat/rgcn-multi-relational' set up to track remote branch 'feat/rgcn-multi-relational' from 'origin'.
Switched to a new branch 'feat/rgcn-multi-relational'
On branch feat/rgcn-multi-relational
Your branch is up to date with 'origin/feat/rgcn-multi-relational'.

nothing to commit, working tree clean


In [3]:
# Cài đặt PyTorch Geometric (hỗ trợ RGCNConv)
!pip install -q torch_geometric

# Tải mô hình cú pháp tiếng Anh của SpaCy
!python -m spacy download en_core_web_sm

# Tải từ điển WordNet (hỗ trợ kiểm thử nhiễu từ đồng nghĩa)
!python -c "import nltk; nltk.download('wordnet'); nltk.download('omw-1.4')"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.7 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 21.0 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


In [4]:
# [CELL 1] Cài đặt thư viện và chuẩn bị môi trường
!pip install -q datasets pandas transformers spacy scikit-learn tqdm
!python -m spacy download en_core_web_sm

import os
os.makedirs("original_text", exist_ok=True)
os.makedirs("graph_data", exist_ok=True)
os.makedirs("result", exist_ok=True)
print("-> Đã cài đặt xong môi trường và tạo các thư mục lưu trữ.")



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.9 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
-> Đã cài đặt xong môi trường và tạo các thư mục lưu trữ.


In [5]:
#[CELL 2]
!python prepare_raid_data.py --seed 2026 --num_train 1000 --num_valid 100 --num_test_per_domain 1000


 BẮT ĐẦU TRÍCH XUẤT VÀ TẠO BỘ DỮ LIỆU BALANCED RAID
- Seed ngẫu nhiên      : 2026
- Thư mục output chính : original_text
- Thư mục test phụ     : 
- Số mẫu Train         : 1000 (50% Human, 50% AI)
- Số mẫu Valid         : 100 (50% Human, 50% AI)
- Số mẫu Test / domain : 1000 (50% Human, 50% AI)
- Các miền lựa chọn    : ['abstracts', 'books', 'news']
[*] Chỉ tiêu tối thiểu mỗi miền: Human ~ 782, mỗi họ AI ~ 326

[+] Đang tải dữ liệu miền 'abstracts'...
    -> Đang đọc shard 1/2: 0.parquet ...
       Số lượng đã gom miền 'abstracts': {'human': 782, 'gpt': 326, 'llama': 326, 'mistral': 326}
    -> Đã đủ chỉ tiêu mẫu cho miền 'abstracts'!

[+] Đang tải dữ liệu miền 'books'...
    -> Đang đọc shard 1/2: 4.parquet ...
       Số lượng đã gom miền 'books': {'human': 0, 'gpt': 326, 'llama': 0, 'mistral': 0}
    -> Đang đọc shard 2/2: 5.parquet ...
       Số lượng đã gom miền 'books': {'human': 0, 'gpt': 326, 'llama': 0, 'mistral': 0}

[+] Đang tải dữ liệu miền 'news'...
    -> Đang đọc shard 1/

In [15]:
# [CELL 3] Xây dựng đồ thị cây cú pháp phụ thuộc cho tất cả các tập
# !python building_graph.py -f mini_raid_test_abstracts mini_raid_test_books mini_raid_test_news mini_raid_test_all
# !python building_graph.py -f hc3_train hc3_val
!python building_graph.py -f hc3_test hc3_val gpt3.5_mixed_test_split gpt3.5_mixed_test
#!python building_graph.py -f raid_llama_greedy_test raid_llama_sampling_test raid_mistral_test



[Processing] Đang xử lý xây dựng đồ thị cho: hc3_test ...
-> Nạp văn bản từ: original_text/hc3_test.json
-> Đã đọc 1000 mẫu văn bản từ file hc3_test
--> Đang nạp mô hình SpaCy (en_core_web_sm) và RoBERTa trên thiết bị: cuda ...
Loading weights: 100%|█| 197/197 [00:00<00:00, 1410.99it/s, Materializing param=
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialize

In [6]:
# [CELL 4] Huấn luyện GCN Baseline và RGCN trên tập HC3 gốc (Batch Size = 32)
print("\n" + "="*70)
print(" 1/2. HUẤN LUYỆN GCN BASELINE TRÊN HC3")
print("="*70)
!python gcn.py --dataset hc3 --train_file hc3_train --val_file hc3_val --batch_size 64 --epochs 25 --seed 2026 --no_test

print("\n" + "="*70)
print(" 2/2. HUẤN LUYỆN RGCN (MULTI-RELATIONAL) TRÊN HC3")
print("="*70)
!python rgcn.py --dataset hc3 --train_file hc3_train --val_file hc3_val --batch_size 64 --epochs 25 --seed 2026 --no_test



 1/2. HUẤN LUYỆN GCN BASELINE TRÊN HC3

 HUẤN LUYỆN MÔ HÌNH GCN BASELINE
 Thiết bị: cuda | Dataset: hc3 | Seed: 2026 | Batch: 64

-> Nạp dữ liệu Train: ./graph_data/hc3_train.pkl
-> Nạp dữ liệu Val:   ./graph_data/hc3_val.pkl
-> Chuẩn bị DataLoader Mini-Batching (batch_size=64) ...
-> Train: 8000 mẫu (125 batches) | Val: 1000 mẫu (16 batches)
Epoch 1/25 [Train GCN]: 100%|█| 125/125 [00:04<00:00, 26.91it/s, loss=0.4037, ac
[Epoch 1] Train Loss: 0.5886 | Train Acc: 0.7218 | Val Loss: 0.4441 | Val Acc: 0.8070
  --> [LƯU MÔ HÌNH TỐT NHẤT] Epoch 1 - Val Acc: 0.8070 -> ./model/hc3_gcn_model_2026.pth
Epoch 2/25 [Train GCN]: 100%|█| 125/125 [00:04<00:00, 28.72it/s, loss=0.2083, ac
[Epoch 2] Train Loss: 0.3529 | Train Acc: 0.8672 | Val Loss: 0.2669 | Val Acc: 0.9250
  --> [LƯU MÔ HÌNH TỐT NHẤT] Epoch 2 - Val Acc: 0.9250 -> ./model/hc3_gcn_model_2026.pth
Epoch 3/25 [Train GCN]: 100%|█| 125/125 [00:04<00:00, 29.04it/s, loss=0.1803, ac
[Epoch 3] Train Loss: 0.2358 | Train Acc: 0.9251 | Val Loss: 

In [12]:
# [CELL DỌN DẸP DISK] Xóa các file train.pkl nặng hàng chục GB để nhường chỗ cho test
!rm -f graph_data/*.pkl
!rm -f graph_data/hc3_val.pkl
!rm -f graph_data/mini_raid_train.pkl
!rm -f graph_data/gpt3.5_mixed_train_split.pkl

print("-> Đã xóa các file Train đồ thị, giải phóng ngay ~12GB bộ nhớ đĩa!")


-> Đã xóa các file Train đồ thị, giải phóng ngay ~12GB bộ nhớ đĩa!


In [13]:
# [CELL 5] Đánh giá trên các tập Test của RAID (Abstracts, Books, News, All)

# [CELL] Đánh giá kiểm thử GCN và RGCN (huấn luyện trên HC3) trên 6 tập dữ liệu

test_sets = ['mini_raid_test_abstracts', 
             'mini_raid_test_books', 
             'mini_raid_test_news', 
             'mini_raid_test_all',
             'raid_llama_greedy_test',
            'raid_llama_sampling_test',
            'raid_llama_test',
            'raid_mistral_test']
for t_file in test_sets:
    print("\n" + "="*70)
    print(f" KIỂM THỬ TẬP: {t_file}")
    print("="*70)
    
    print(f"\n>>> [1/2] {t_file} với GCN Baseline:")
    !python test.py --dataset hc3 --file {t_file} --model_type gcn --batch_size 32 --seed 2026 -s
    
    print(f"\n>>> [2/2] {t_file} với RGCN (Đa quan hệ):")
    !python test.py --dataset hc3 --file {t_file} --model_type rgcn --batch_size 32 --seed 2026 -s



 KIỂM THỬ TẬP: mini_raid_test_abstracts

>>> [1/2] mini_raid_test_abstracts với GCN Baseline:
Testing: file=mini_raid_test_abstracts, dataset=hc3, seed=2026, model=gcn, batch_size=32
Test (GCN Batch=32): 100%|██████████████████████| 32/32 [00:01<00:00, 28.10it/s]
Elapsed time: 1.14 seconds (1.14 ms/sample)
Model: GCN | test_loss: 1.5975, test_acc: 0.7047, test_f1: 0.7607, auc: 0.6596

>>> [2/2] mini_raid_test_abstracts với RGCN (Đa quan hệ):
Testing: file=mini_raid_test_abstracts, dataset=hc3, seed=2026, model=rgcn, batch_size=32
Test (RGCN Batch=32): 100%|█████████████████████| 32/32 [00:04<00:00,  7.45it/s]
Elapsed time: 4.30 seconds (4.30 ms/sample)
Model: RGCN | test_loss: 1.2822, test_acc: 0.7037, test_f1: 0.7574, auc: 0.6709

 KIỂM THỬ TẬP: mini_raid_test_books

>>> [1/2] mini_raid_test_books với GCN Baseline:
Testing: file=mini_raid_test_books, dataset=hc3, seed=2026, model=gcn, batch_size=32
Test (GCN Batch=32): 100%|████████████████████████| 6/6 [00:00<00:00, 14.99it/s]
Elaps

In [16]:
# [CELL 6] Đánh giá Out-of-Domain trên tập HC3 gốc và GPT-3.5 Mixed
ood_sets = ['hc3_test', 'gpt3.5_mixed_test_split' ]

for t_file in ood_sets:
    print(f"\n>>> [CROSS-DATASET TEST] {t_file} với GCN Baseline:")
    !python test.py --dataset hc3 --file {t_file} --seed 2026 --model_type gcn --batch_size 32 -s

    print(f"\n>>> [CROSS-DATASET TEST] {t_file} với RGCN (Đa quan hệ):")
    !python test.py --dataset hc3 --file {t_file} --seed 2026 --model_type rgcn --batch_size 32 -s



>>> [CROSS-DATASET TEST] hc3_test với GCN Baseline:
Testing: file=hc3_test, dataset=hc3, seed=2026, model=gcn, batch_size=32
Test (GCN Batch=32): 100%|██████████████████████| 32/32 [00:00<00:00, 45.53it/s]
Elapsed time: 0.70 seconds (0.70 ms/sample)
Model: GCN | test_loss: 0.0790, test_acc: 0.9740, test_f1: 0.9746, auc: 0.9960

>>> [CROSS-DATASET TEST] hc3_test với RGCN (Đa quan hệ):
Testing: file=hc3_test, dataset=hc3, seed=2026, model=rgcn, batch_size=32
Test (RGCN Batch=32): 100%|█████████████████████| 32/32 [00:03<00:00,  9.52it/s]
Elapsed time: 3.36 seconds (3.36 ms/sample)
Model: RGCN | test_loss: 0.0545, test_acc: 0.9860, test_f1: 0.9861, auc: 0.9979

>>> [CROSS-DATASET TEST] gpt3.5_mixed_test_split với GCN Baseline:
Testing: file=gpt3.5_mixed_test_split, dataset=hc3, seed=2026, model=gcn, batch_size=32
Test (GCN Batch=32): 100%|██████████████████████| 32/32 [00:02<00:00, 12.77it/s]
Elapsed time: 2.51 seconds (2.51 ms/sample)
Model: GCN | test_loss: 1.0701, test_acc: 0.6490, te

In [17]:
# [CELL 7] Bảng So sánh Tổng quan Hiệu năng (GCN vs RGCN) - Chỉ lọc kết quả Seed 2026
import os
import re
import pandas as pd
from IPython.display import display

TARGET_SEED = "2026"
records = []

if os.path.exists("test_result.txt"):
    with open("test_result.txt", "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 6:
                test_file = parts[0].strip()
                acc_match   = re.search(r"acc:\s*([\d\.]+)", line)
                auc_match   = re.search(r"auc:\s*([\d\.]+)", line)
                f1_match    = re.search(r"f1:\s*([\d\.]+)", line)
                seed_match  = re.search(r"seed:\s*(\d+)", line)
                model_match = re.search(r"model:\s*([^\t]+)", line)
                
                # CHỈ LẤY CÁC DÒNG CÓ SEED 2026 VÀ CỦA MODEL MINI_RAID
                if acc_match and auc_match and f1_match and seed_match and model_match:
                    line_seed = seed_match.group(1).strip()
                    model_str = model_match.group(1).strip()
                    
                    if line_seed == TARGET_SEED and "hc3" in model_str:
                        m_type = "RGCN" if "rgcn" in model_str else "GCN"
                        records.append({
                            "Test Dataset": test_file,
                            "Model": m_type,
                            "Accuracy (%)": round(float(acc_match.group(1)) * 100, 2),
                            "ROC-AUC": round(float(auc_match.group(1)), 4),
                            "F1-Score": round(float(f1_match.group(1)), 4)
                        })

if records:
    df = pd.DataFrame(records).drop_duplicates(subset=["Test Dataset", "Model"], keep="last")
    pivot_acc = df.pivot(index="Test Dataset", columns="Model", values="Accuracy (%)")
    pivot_auc = df.pivot(index="Test Dataset", columns="Model", values="ROC-AUC")
    pivot_f1  = df.pivot(index="Test Dataset", columns="Model", values="F1-Score")
    
    summary_df = pd.DataFrame({
        "GCN Acc (%)": pivot_acc.get("GCN", 0),
        "RGCN Acc (%)": pivot_acc.get("RGCN", 0),
        "Δ Acc": pivot_acc.get("RGCN", 0) - pivot_acc.get("GCN", 0),
        "GCN AUC": pivot_auc.get("GCN", 0),
        "RGCN AUC": pivot_auc.get("RGCN", 0),
        "Δ AUC": pivot_auc.get("RGCN", 0) - pivot_auc.get("GCN", 0),
        "GCN F1": pivot_f1.get("GCN", 0),
        "RGCN F1": pivot_f1.get("RGCN", 0),
        "Δ F1": pivot_f1.get("RGCN", 0) - pivot_f1.get("GCN", 0)
    }).reset_index()
    
    print("\n" + "="*85)
    print(f" BẢNG SO SÁNH TỔNG QUAN HIỆU NĂNG: GCN BASELINE VS RGCN (SEED {TARGET_SEED})")
    print("="*85)
    display(summary_df)
else:
    print(f"Chưa tìm thấy kết quả nào với seed {TARGET_SEED} trong test_result.txt.")
    print("Hãy đảm bảo bạn đã thêm '--seed 2026' khi chạy test ở Cell 5 và Cell 6.")



 BẢNG SO SÁNH TỔNG QUAN HIỆU NĂNG: GCN BASELINE VS RGCN (SEED 2026)


,Test Dataset,GCN Acc (%),RGCN Acc (%),Δ Acc,GCN AUC,RGCN AUC,Δ AUC,GCN F1,RGCN F1,Δ F1
0,gpt3.5_mixed_test_split,64.90,72.60,7.70,0.8614,0.8782,0.0168,0.7347,0.7794,0.0447
1,hc3_test,97.40,98.60,1.20,0.9960,0.9979,0.0019,0.9746,0.9861,0.0115
2,mini_raid_test_abstracts,70.47,70.37,-0.10,0.6596,0.6709,0.0113,0.7607,0.7574,-0.0033
3,mini_raid_test_all,68.59,70.39,1.80,0.7176,0.7310,0.0134,0.7375,0.7475,0.0100
4,mini_raid_test_news,67.27,68.57,1.30,0.7093,0.7148,0.0055,0.7483,0.7562,0.0079
5,raid_llama_greedy_test,74.40,76.80,2.40,0.8875,0.8617,-0.0258,0.7836,0.7945,0.0109
6,raid_llama_sampling_test,73.05,74.50,1.45,0.8743,0.8491,-0.0252,0.7748,0.7786,0.0038
7,raid_mistral_test,78.10,78.40,0.30,0.9080,0.8788,-0.0292,0.8089,0.8059,-0.0030


In [6]:
# ==============================================================================
# SCRIPT ĐỘC LẬP: THỐNG KÊ CHI TIẾT ĐỘ DÀI DỮ LIỆU ĐÃ CÀO BỞI prepare_raid_data.py
# ==============================================================================
import os
import json
import statistics
import pandas as pd
from IPython.display import display

# Danh sách các file dữ liệu được tạo bởi prepare_raid_data.py
dataset_files = {
    "Train (8k)": "mini_raid_train.json",
    "Valid (1k)": "mini_raid_val.json",
    "Test_Abstracts (1k)": "mini_raid_test_abstracts.json",
    "Test_Books (1k)": "mini_raid_test_books.json",
    "Test_News (1k)": "mini_raid_test_news.json",
    "Test_All (3k)": "mini_raid_test_all.json"
}

# Các thư mục tìm kiếm tiềm năng
search_dirs = ["original_text", "data", "."]

loaded_datasets = {}
train_records = []

for split_name, fname in dataset_files.items():
    fpath = None
    for d in search_dirs:
        for ext in [".json", ".jsonl"]:
            cand = os.path.join(d, fname.replace(".json", ext))
            if os.path.exists(cand):
                fpath = cand
                break
        if fpath:
            break
            
    if not fpath:
        continue
        
    samples = []
    with open(fpath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                samples.append(item)
                if split_name == "Train (8k)":
                    train_records.append(item)
                    
    loaded_datasets[split_name] = samples

if not loaded_datasets:
    print("[!] Chưa tìm thấy các file dữ liệu 'mini_raid_*.json'.")
    print("    Vui lòng kiểm tra xem bạn đã chạy prepare_raid_data.py chưa.")
else:
    # --------------------------------------------------------------------------
    # 1. BẢNG THỐNG KÊ ĐỘ DÀI TỪNG TẬP DỮ LIỆU
    # --------------------------------------------------------------------------
    table1_rows = []
    for split_name, samples in loaded_datasets.items():
        w_counts = [len(str(s.get("text", "")).split()) for s in samples]
        c_counts = [len(str(s.get("text", ""))) for s in samples]
        n = len(w_counts)
        
        table1_rows.append({
            "Tập Dữ Liệu": split_name,
            "Số mẫu": n,
            "Mean (từ)": round(statistics.mean(w_counts), 1),
            "Std (từ)": round(statistics.stdev(w_counts), 1) if n > 1 else 0.0,
            "Median (từ)": round(statistics.median(w_counts), 1),
            "Min (từ)": min(w_counts),
            "Max (từ)": max(w_counts),
            "Mean (ký tự)": round(statistics.mean(c_counts), 1)
        })
        
    df_table1 = pd.DataFrame(table1_rows)
    print("=" * 95)
    print(" BẢNG 1: THỐNG KÊ ĐỘ DÀI VĂN BẢN (SỐ TỪ & KÝ TỰ) THEO TỪNG TẬP CÀO VỀ")
    print("=" * 95)
    display(df_table1)

    # --------------------------------------------------------------------------
    # 2. KIỂM TRA THIÊN LỆCH ĐỘ DÀI: HUMAN VS CÁC MÔ HÌNH AI TRONG TẬP TRAIN
    # --------------------------------------------------------------------------
    if train_records:
        fam_groups = {}
        for s in train_records:
            fam = str(s.get("family", "Human" if s.get("label") == 0 else "AI")).upper()
            fam_groups.setdefault(fam, []).append(len(str(s.get("text", "")).split()))
            
        table2_rows = []
        for fam in sorted(fam_groups.keys()):
            w_list = fam_groups[fam]
            n = len(w_list)
            table2_rows.append({
                "Nhóm Tác giả / Mô hình": fam,
                "Số mẫu": n,
                "Mean (từ)": round(statistics.mean(w_list), 1),
                "Median (từ)": round(statistics.median(w_list), 1),
                "Min (từ)": min(w_list),
                "Max (từ)": max(w_list)
            })
            
        df_table2 = pd.DataFrame(table2_rows)
        print("\n" + "=" * 95)
        print(" BẢNG 2: KIỂM TRA THIÊN LỆCH ĐỘ DÀI (LENGTH BIAS) HUMAN VS AI TRONG TẬP TRAIN")
        print("=" * 95)
        display(df_table2)

        # --------------------------------------------------------------------------
        # 3. THỐNG KÊ ĐỘ DÀI THEO TỪNG MIỀN VĂN BẢN (DOMAIN)
        # --------------------------------------------------------------------------
        dom_groups = {}
        for s in train_records:
            dom = str(s.get("domain", "unknown"))
            dom_groups.setdefault(dom, []).append(len(str(s.get("text", "")).split()))
            
        table3_rows = []
        for dom in sorted(dom_groups.keys()):
            w_list = dom_groups[dom]
            n = len(w_list)
            table3_rows.append({
                "Miền Văn Bản (Domain)": dom,
                "Số mẫu": n,
                "Mean (từ)": round(statistics.mean(w_list), 1),
                "Median (từ)": round(statistics.median(w_list), 1),
                "Min (từ)": min(w_list),
                "Max (từ)": max(w_list)
            })
            
        df_table3 = pd.DataFrame(table3_rows)
        print("\n" + "=" * 95)
        print(" BẢNG 3: ĐỘ DÀI TRUNG BÌNH THEO MIỀN VĂN BẢN (DOMAIN)")
        print("=" * 95)
        display(df_table3)


 BẢNG 1: THỐNG KÊ ĐỘ DÀI VĂN BẢN (SỐ TỪ & KÝ TỰ) THEO TỪNG TẬP CÀO VỀ


,Tập Dữ Liệu,Số mẫu,Mean (từ),Std (từ),Median (từ),Min (từ),Max (từ),Mean (ký tự)
0,Train (8k),720,257.4,194.0,224.0,43,3545,1535.0
1,Valid (1k),70,252.5,208.0,213.0,66,1804,1503.6
2,Test_Abstracts (1k),999,232.5,86.8,211.0,77,453,1571.1
3,Test_Books (1k),167,129.7,67.7,114.0,32,419,726.0
4,Test_News (1k),999,321.4,620.6,238.0,36,13158,1733.0
5,Test_All (3k),2165,265.6,429.9,219.0,32,13158,1580.6



 BẢNG 2: KIỂM TRA THIÊN LỆCH ĐỘ DÀI (LENGTH BIAS) HUMAN VS AI TRONG TẬP TRAIN


,Nhóm Tác giả / Mô hình,Số mẫu,Mean (từ),Median (từ),Min (từ),Max (từ)
0,GPT,168,263.7,281.5,43,478
1,HUMAN,332,264.4,211.0,99,3545
2,LLAMA,110,230.9,240.5,107,359
3,MISTRAL,110,253.4,265.0,82,425



 BẢNG 3: ĐỘ DÀI TRUNG BÌNH THEO MIỀN VĂN BẢN (DOMAIN)


,Miền Văn Bản (Domain),Số mẫu,Mean (từ),Median (từ),Min (từ),Max (từ)
0,abstracts,332,232.3,212.0,82,451
1,books,56,139.4,122.5,43,409
2,news,332,302.4,247.0,81,3545


In [7]:
# [CELL SỬA NHÃN CẤP TỐC] Chuẩn hóa nhãn mini_raid thành 'human' và 'machine'
import glob
import json

files_to_fix = glob.glob("original_text/mini_raid_*.json")
print(f"-> Đang sửa nhãn cho {len(files_to_fix)} file...")

for fpath in files_to_fix:
    fixed_lines = []
    with open(fpath, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            # Chuẩn hóa: Nếu là Human thì gán "human", nếu là AI thì gán "machine"
            if item.get("family") == "human" or item.get("label") == 0:
                item["label"] = "human"
            else:
                item["label"] = "machine"
            fixed_lines.append(json.dumps(item, ensure_ascii=False))
            
    with open(fpath, "w", encoding="utf-8") as f:
        f.write("\n".join(fixed_lines) + "\n")
    print(f"   [OK] Đã sửa nhãn chuẩn cho: {fpath}")

print("\n-> Bắt đầu build lại đồ thị cây cú pháp (chỉ mất ~1-2 phút)...")
!python building_graph.py -f mini_raid_test_abstracts mini_raid_test_books mini_raid_test_news mini_raid_test_all


-> Đang sửa nhãn cho 6 file...
   [OK] Đã sửa nhãn chuẩn cho: original_text/mini_raid_test_abstracts.json
   [OK] Đã sửa nhãn chuẩn cho: original_text/mini_raid_test_books.json
   [OK] Đã sửa nhãn chuẩn cho: original_text/mini_raid_test_news.json
   [OK] Đã sửa nhãn chuẩn cho: original_text/mini_raid_train.json
   [OK] Đã sửa nhãn chuẩn cho: original_text/mini_raid_val.json
   [OK] Đã sửa nhãn chuẩn cho: original_text/mini_raid_test_all.json

-> Bắt đầu build lại đồ thị cây cú pháp (chỉ mất ~1-2 phút)...

[Processing] Đang xử lý xây dựng đồ thị cho: mini_raid_test_abstracts ...
-> Nạp văn bản từ: original_text/mini_raid_test_abstracts.json
-> Đã đọc 999 mẫu văn bản từ file mini_raid_test_abstracts
--> Đang nạp mô hình SpaCy (en_core_web_sm) và RoBERTa trên thiết bị: cuda ...
config.json: 100%|█████████████████████████████| 481/481 [00:00<00:00, 1.23MB/s]
model.safetensors: 100%|██████████████████████| 499M/499M [00:02<00:00, 187MB/s]
Loading weights: 100%|█| 197/197 [00:00<00:00, 1494.

In [10]:
!cp -r /kaggle/input/datasets/brokensw1402/trongsohc3/hc3_rgcn_model_2026.pth /kaggle/working/PRDetect-Research/model